In [5]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/metadata/source-manifest.csv")
df.head()

,source_id,reporting_month,filename,relative_path,report_section,fact_sheet_format,file_status,extraction_status,review_status,notes
0,SRC_0001,2024-01,operational-performance-jan-2024_converted.pdf,data/raw/nerc/operational-performance-jan-2024...,combined,old,available,not_started,not_reviewed,NaN
1,SRC_0002,2024-02,Operational-performance-Feb-2024_converted.pdf,data/raw/nerc/Operational-performance-Feb-2024...,combined,old,available,not_started,not_reviewed,NaN
2,SRC_0003,2024-03,Operational-performance-Mar-2024_converted.pdf,data/raw/nerc/Operational-performance-Mar-2024...,combined,old,available,not_started,not_reviewed,NaN
3,SRC_0004,2024-04,Operational-Performance-Factsheet-April-2024_c...,data/raw/nerc/Operational-Performance-Factshee...,combined,old,available,not_started,not_reviewed,NaN
4,SRC_0005,2024-05,Operational-performance-factsheet-May-2024_con...,data/raw/nerc/Operational-performance-factshee...,combined,old,available,not_started,not_reviewed,NaN


In [ ]:
repr_src_ids = ["SRC_0001", "SRC_0020", "SRC_0021"]

In [ ]:
df_repr_sources = df[df.source_id.isin(repr_src_ids)].reset_index(drop=True)
df_repr_sources

In [6]:
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path.cwd().parent

In [ ]:
df_repr_sources['absolute_path'] = df_repr_sources.relative_path.apply(
  lambda path: Path(PROJECT_ROOT / path).resolve())
df_repr_sources['file_exists'] = df_repr_sources.absolute_path.apply(
  lambda path: Path(path).is_file())
df_repr_sources[['source_id', 'filename', 'file_exists']]

In [7]:
import pymupdf

In [ ]:
for _, source in df_repr_sources.iterrows():
  pdf_path = source['absolute_path']
  
  with pymupdf.open(pdf_path) as pdf:
    print(f"Source: {source.source_id}")
    print(f"Pages: {len(pdf)}")
    
    for page_number, page in enumerate(pdf):
      text = page.get_text("text")
      print(
        f"Page {page_number + 1}: "
        f"{len(text.strip())} extracted characters"
      )
      with pymupdf.open(pdf_path) as pdf:
        page2 = pdf[page_number]
        pix = page.get_pixmap(matrix=pymupdf.Matrix(2,2))
        pix.save(f"../data/interim/inpection-{source.source_id}.png")

In [8]:
import numpy as np
from rapidocr import RapidOCR

engine = RapidOCR()

In [ ]:
pdf_path = df_repr_sources.absolute_path[0]
doc = pymupdf.open("../data/raw/nerc/Operational-Performance-Factsheet-Aug-2025_2.pdf")
md_text = doc.to_markdown(force_ocr=True)

In [ ]:
print(pymupdf.__version__)

In [ ]:
print(md_text)

In [ ]:
output_dir = Path("../data/interim/markdown")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
for _, source in df_repr_sources.iterrows():
  pdf_path = source.absolute_path
  doc = pymupdf.open(pdf_path)
  md_text = doc.to_markdown(force_ocr=True)
  md_filename = source.filename.replace(".pdf", ".md")
  output_path = output_dir / md_filename
  source['md_filename'] = md_filename
  output_path.write_text(md_text, encoding="utf-8")

# Extraction from OCR generated markdown file

In [9]:
import re

In [6]:
md_new_grid_path = Path("../data/interim/markdown/Operational-Performance-Factsheet-Aug-2025_1.md")
md_new_plant_path = Path("../data/interim/markdown/Operational-Performance-Factsheet-Aug-2025_2.md")

## Initial extraction for grid data for the new format documents

In [7]:
md_new_grid_text = md_new_grid_path.read_text(encoding="utf-8")
md_new_grid_text[:10000]

'\n\n<!-- Start of picture text -->\nNigerian Electricity<br>Regulatory Commission<br><!-- End of picture text -->\n\n\n\n<!-- Start of picture text -->\nOperational Performance<br>of Power Plants 1/2<br><!-- End of picture text -->\n\n\n\n<!-- Start of picture text -->\nFACTSHEET<br>August, 2025<br><!-- End of picture text -->\n\n\n\n<!-- Start of picture text -->\nGrid Performance<br>[c] Voltage Stability<br>313.50 346.50 350.77<br>300.65<br>- Target Voltage Range<br>- Minimum Voltage Recorded<br>- Maximum Voltage Recorded<br>kV<br><!-- End of picture text -->\n\n\n\n<!-- Start of picture text -->\nEnergy Generation<br><!-- End of picture text -->\n\n[A] \n\n\n\n<!-- Start of picture text -->\n[A] Plant Availability<br>5,5141% (Available)<br>MW<br>40%1%<br>Plant Availability<br>Factor<br>0 13,625 (Installed)<br><!-- End of picture text -->\n\nThe grid-connected plants recorded a Plant Availability Factor (PAF) of 40% - At any point during the month, an average of 5,514MW was availabl

In [11]:
for line_number, line in enumerate(md_new_grid_text.splitlines(), start=1):
  print(f"{line_number}: {line}")

1: 
2: 
3: <!-- Start of picture text -->
4: Nigerian Electricity<br>Regulatory Commission<br><!-- End of picture text -->
5: 
6: 
7: 
8: <!-- Start of picture text -->
9: Operational Performance<br>of Power Plants 1/2<br><!-- End of picture text -->
10: 
11: 
12: 
13: <!-- Start of picture text -->
14: FACTSHEET<br>August, 2025<br><!-- End of picture text -->
15: 
16: 
17: 
18: <!-- Start of picture text -->
19: Grid Performance<br>[c] Voltage Stability<br>313.50 346.50 350.77<br>300.65<br>- Target Voltage Range<br>- Minimum Voltage Recorded<br>- Maximum Voltage Recorded<br>kV<br><!-- End of picture text -->
20: 
21: 
22: 
23: <!-- Start of picture text -->
24: Energy Generation<br><!-- End of picture text -->
25: 
26: [A] 
27: 
28: 
29: 
30: <!-- Start of picture text -->
31: [A] Plant Availability<br>5,5141% (Available)<br>MW<br>40%1%<br>Plant Availability<br>Factor<br>0 13,625 (Installed)<br><!-- End of picture text -->
32: 
33: The grid-connected plants recorded a Plant Availabili

In [12]:
df_grid = pd.DataFrame(columns=[
  'source_id',
  'reporting_month',
  'lower_voltage',
  'upper_voltage',
  'lower_frequency',
  'upper_frequency'
])

In [13]:
grid_idx = len(df_grid)

df_grid.loc[grid_idx, ['source_id', 'reporting_month']] = ['SRC_0021', '2025-08']

In [14]:
df_grid.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   source_id        1 non-null      object
 1   reporting_month  1 non-null      object
 2   lower_voltage    0 non-null      object
 3   upper_voltage    0 non-null      object
 4   lower_frequency  0 non-null      object
 5   upper_frequency  0 non-null      object
dtypes: object(6)
memory usage: 180.0+ bytes


In [15]:
voltage_pattern = r"average lower grid voltage \((\d+\.\d+kV)\) and average upper grid voltage \((\d+\.\d+kV)\)"

for line_number, line in enumerate(md_new_grid_text.splitlines(), start=1):
  voltage_match = re.search(voltage_pattern, line)
  if voltage_match:
    lower_voltage = voltage_match.group(1)
    upper_voltage = voltage_match.group(2)
    df_grid.loc[grid_idx, ['lower_voltage', 'upper_voltage']] = voltage_match.groups()

In [16]:
df_grid

,source_id,reporting_month,lower_voltage,upper_voltage,lower_frequency,upper_frequency
0,SRC_0021,2025-08,300.65kV,350.77kV,NaN,NaN


In [17]:
freq_pattern = r"average lower grid frequency \((\d+\.\d+Hz)\) and average upper grid frequency \((\d+\.\d+Hz)\)"

for line_number, line in enumerate(md_new_grid_text.splitlines(), start=1):
  freq_match = re.search(freq_pattern, line)
  if freq_match:
    lower_freq = freq_match.group(1)
    upper_freq = freq_match.group(2)
    df_grid.loc[grid_idx, ['lower_frequency', 'upper_frequency']] = [lower_freq, upper_freq]

In [18]:
df_grid

,source_id,reporting_month,lower_voltage,upper_voltage,lower_frequency,upper_frequency
0,SRC_0021,2025-08,300.65kV,350.77kV,49.37Hz,50.78Hz


In [19]:
df_grid.to_csv(
    "../data/interim/extracted/Operational-Performance-Factsheet-Aug-2025_1.csv", index=False)

## Initial extraction for the plant data for the new document format

In [8]:
md_new_plant_text = md_new_plant_path.read_text(encoding="utf8")
md_new_plant_text[:1000]

'Nigerian Electricity Regulatory Commission \n\n# Operational Performance of Power Plants 2/2 \n\n\n\n<!-- Start of picture text -->\nFACTSHEET<br><!-- End of picture text -->\n\nAugust, 2025 \n\n\n\n<!-- Start of picture text -->\nLargest Energy Producers - August 2025<br>The top ten largest energy producers accounted for 79% of the total energy<br>generated during the month.<br>52%(690/1320MW) Egbin_1 93% (640MWh/h)<br>56% (507/900MW) Detl 85% (429MWh/h)<br>52%(397/760MW) Kainji_1 94% (374MWh/h)<br>97% (681/700MW) Zungeru_1 43% (292MWh/h)<br>31% (203/650MW) Afam_2 97% (196MWh/h)<br>73% (435/600MW) Shiroro_1 60% (259MWh/h)<br>82% (472/578MW) Jebba_1 73% (344MWh/h)<br>66% (319/480MW) Okpai_1 78% (249MWh/h)<br>72% (332/461MW) Ihovbor_2 91% (302MWh/h)<br>54% (235/435MW) Geregu_1 71% (167MWh/h)<br>%001 80 60 40 20 0 20 40 60 80 100%<br>Plant Availability Factor Load Factor<br><!-- End of picture text -->\n\n||Other Gr|id Connecte<br>Aerae|d Plants<br>|||\n|---|---|---|---|---|---|\n|Plant

In [20]:
for line_number, line in enumerate(md_new_plant_text.splitlines(), start=1):
  print(f"{line_number}: {line}")

1: Nigerian Electricity Regulatory Commission 
2: 
3: # Operational Performance of Power Plants 2/2 
4: 
5: 
6: 
7: <!-- Start of picture text -->
8: FACTSHEET<br><!-- End of picture text -->
9: 
10: August, 2025 
11: 
12: 
13: 
14: <!-- Start of picture text -->
15: Largest Energy Producers - August 2025<br>The top ten largest energy producers accounted for 79% of the total energy<br>generated during the month.<br>52%(690/1320MW) Egbin_1 93% (640MWh/h)<br>56% (507/900MW) Detl 85% (429MWh/h)<br>52%(397/760MW) Kainji_1 94% (374MWh/h)<br>97% (681/700MW) Zungeru_1 43% (292MWh/h)<br>31% (203/650MW) Afam_2 97% (196MWh/h)<br>73% (435/600MW) Shiroro_1 60% (259MWh/h)<br>82% (472/578MW) Jebba_1 73% (344MWh/h)<br>66% (319/480MW) Okpai_1 78% (249MWh/h)<br>72% (332/461MW) Ihovbor_2 91% (302MWh/h)<br>54% (235/435MW) Geregu_1 71% (167MWh/h)<br>%001 80 60 40 20 0 20 40 60 80 100%<br>Plant Availability Factor Load Factor<br><!-- End of picture text -->
16: 
17: ||Other Gr|id Connecte<br>Aerae|d Plants

In [21]:
df_plant = pd.DataFrame(columns=[
  'source_id',
  'line_number',
  'reporting_month',
  'plant_name',
  'installed_capacity',
  'available_capacity',
  'availability_factor',
  'average_generation',
  'total_generation',
  'load_factor'
])

In [ ]:
top_10_pattern = r"^Largest Energy Producers"
top_10_line_pattern = r"(\d+%)\s*\((\d+)/(\d+MW)\)\s+(\S+)\s+(\d+%)\s+\((\d+MWh/h)\)"
data_pattern = (
  r"\|(?P<plant_name>[^|]*)\|"
  r"(?P<installed_capacity>[\d,]+)\|"
  r"(?P<available_capacity>[\d,]+)\|"
  r"(?P<availability_factor>[^|]+)\|"
  r"(?P<average_generation>[\d,]+)\|"
  r"(?P<load_factor>[^|]+)\|"
)

rows = []

for line_number, line in enumerate(md_new_plant_text.splitlines()):
  top_10_line = re.search(top_10_pattern, line)
  data_line = re.search(data_pattern, line)
  if top_10_line:
    top_10 = line.split('<br>')
    for top_10_line in top_10:
      line_match = re.search(top_10_line_pattern, top_10_line)
      if line_match:
        rows.append({
          'source_id': 'SRC_0021',
          'line_number': line_number,
          "reporting_month": '2025-08',
          "plant_name": line_match.group(4),
          "installed_capacity": line_match.group(3),
          "available_capacity": line_match.group(2),
          "availability_factor": line_match.group(1),
          "average_generation": line_match.group(6),
          "load_factor": line_match.group(5),
        })
  elif data_line:
    rows.append({
      'source_id': 'SRC_0021',
      'line_number': line_number,
      "reporting_month": '2025-08',
      **data_line.groupdict()
    })

df_plant = pd.concat([df_plant, pd.DataFrame(rows)], ignore_index=True)

In [23]:
df_plant.head()

,source_id,line_number,reporting_month,plant_name,installed_capacity,available_capacity,availability_factor,average_generation,total_generation,load_factor
0,SRC_0021,14,2025-08,Egbin_1,1320MW,690,52%,640MWh/h,NaN,93%
1,SRC_0021,14,2025-08,Detl,900MW,507,56%,429MWh/h,NaN,85%
2,SRC_0021,14,2025-08,Kainji_1,760MW,397,52%,374MWh/h,NaN,94%
3,SRC_0021,14,2025-08,Zungeru_1,700MW,681,97%,292MWh/h,NaN,43%
4,SRC_0021,14,2025-08,Afam_2,650MW,203,31%,196MWh/h,NaN,97%


In [89]:
df_plant.to_csv('../data/interim/extracted/Operational-Performance-Factsheet-Aug-2025_2.csv', index=False)

## Initial extraction for the grid data for old document formats

In [10]:
md_old_path = Path("../data/interim/markdown/operational-performance-jan-2024_converted.md")

In [11]:
md_old_text = md_old_path.read_text(encoding="utf-8")
md_old_text[:1000]

'|Nigerian Electricity<br>Regulatory Commission<br>NERC||Operati<br>of|onal Pe<br>Power|rformance<br>Plants||||F|ACTSH<br>January, 2|EET<br>024|\n|---|---|---|---|---|---|---|---|---|---|---|\n|Energy Generation||Grid P|erforman|ce|O|ther Grid|Connec<br>|ted Pla|nts||\n|Grid-Connected Plants<br>26|||Target|Actual|Plant|Capacity<br>Installed<br>(MW)|Available<br>Capacity<br>Average<br>(MW)<br>A<br>F|vailabilit<br>actor (%<br>Plant|y<br>)<br>Generation<br>(GWh)<br>Total<br>|Factor<br>Load<br>(%)|\n|4,459.41MW▼5.66%<br>Total Available Capacity|Grid Fr|upper <br>equency<br>lower|50.25Hz <br>4975Hz|50.62Hz▲0.70%<br>4903Hz▼145%|Afam IV - V<br>Sapele ST|726<br>720|43.97<br>32.74|6%<br>5%|34.98 <br>16.79|100%<br> %66|\n|<br>|||.|..|Olorunsogo NIPP|<br>690|94.77|14%|73.82|100%|\n|4,274.80MWh/h▼3.94%<br>Average Hourly Generation|||k|35080kV▲124%|Omotosho NIPP|500|42.26|8%|26.57|85%|\n|<br>|Grid V|upper<br>oltage|46.50|..|Sapele GT NIPP|452|58.90|13%|46.34|<sup>100%</sup>|\n|3,180.45GWh▼3.94%<br>

In [28]:
for line_number, line in enumerate(md_old_text.splitlines()):
  print(f"{line_number}: {line}")

0: |Nigerian Electricity<br>Regulatory Commission<br>NERC||Operati<br>of|onal Pe<br>Power|rformance<br>Plants||||F|ACTSH<br>January, 2|EET<br>024|
1: |---|---|---|---|---|---|---|---|---|---|---|
2: |Energy Generation||Grid P|erforman|ce|O|ther Grid|Connec<br>|ted Pla|nts||
3: |Grid-Connected Plants<br>26|||Target|Actual|Plant|Capacity<br>Installed<br>(MW)|Available<br>Capacity<br>Average<br>(MW)<br>A<br>F|vailabilit<br>actor (%<br>Plant|y<br>)<br>Generation<br>(GWh)<br>Total<br>|Factor<br>Load<br>(%)|
4: |4,459.41MW▼5.66%<br>Total Available Capacity|Grid Fr|upper <br>equency<br>lower|50.25Hz <br>4975Hz|50.62Hz▲0.70%<br>4903Hz▼145%|Afam IV - V<br>Sapele ST|726<br>720|43.97<br>32.74|6%<br>5%|34.98 <br>16.79|100%<br> %66|
5: |<br>|||.|..|Olorunsogo NIPP|<br>690|94.77|14%|73.82|100%|
6: |4,274.80MWh/h▼3.94%<br>Average Hourly Generation|||k|35080kV▲124%|Omotosho NIPP|500|42.26|8%|26.57|85%|
7: |<br>|Grid V|upper<br>oltage|46.50|..|Sapele GT NIPP|452|58.90|13%|46.34|<sup>100%</sup>|
8: |3,1

In [29]:
md_old_text_lines = md_old_text.splitlines()
line1_grid = md_old_text_lines[0].split("|")
for line in line1_grid:
  print(f"'{line}'")

''
'Nigerian Electricity<br>Regulatory Commission<br>NERC'
''
'Operati<br>of'
'onal Pe<br>Power'
'rformance<br>Plants'
''
''
''
'F'
'ACTSH<br>January, 2'
'EET<br>024'
''


In [30]:
stitched_lines = []
for line in line1_grid:
  if line.strip() == "":
    for i in range(len(stitched_lines)):
      stitched_lines[i] += " "
  else:
    words = line.strip().split("<br>")
    for i in range(len(words)):
      if i >= len(stitched_lines):
        stitched_lines.append(words[i])
      else:
        stitched_lines[i] += words[i]

for sentence in stitched_lines: print(sentence)

Nigerian Electricity Operational Performance   FACTSHEET 
Regulatory Commission ofPowerPlants   January, 2024 
NERC     


In [31]:
line3_grid = md_old_text_lines[2].split("|")
for line in line3_grid:
  print(f"'{line.strip()}'")

''
'Energy Generation'
''
'Grid P'
'erforman'
'ce'
'O'
'ther Grid'
'Connec<br>'
'ted Pla'
'nts'
''
''


In [32]:
stitched_lines = []
for line in line3_grid:
  if line.strip() == "":
    for i in range(len(stitched_lines)):
      stitched_lines[i] += " "
  else:
    words = line.strip().split("<br>")
    for i in range(len(words)):
      if i >= len(stitched_lines):
        stitched_lines.append(words[i])
      else:
        stitched_lines[i] += words[i]
        
for sentence in stitched_lines: print(sentence)

Energy Generation Grid PerformanceOther GridConnected Plants  
  


In [33]:
# Isolate grid rows from the markdown text

grid_lines = md_old_text.splitlines()[3:9]
for line_number, line in enumerate(grid_lines, start=1):
  print(f"{line_number}: {line.strip()}")

1: |Grid-Connected Plants<br>26|||Target|Actual|Plant|Capacity<br>Installed<br>(MW)|Available<br>Capacity<br>Average<br>(MW)<br>A<br>F|vailabilit<br>actor (%<br>Plant|y<br>)<br>Generation<br>(GWh)<br>Total<br>|Factor<br>Load<br>(%)|
2: |4,459.41MW▼5.66%<br>Total Available Capacity|Grid Fr|upper <br>equency<br>lower|50.25Hz <br>4975Hz|50.62Hz▲0.70%<br>4903Hz▼145%|Afam IV - V<br>Sapele ST|726<br>720|43.97<br>32.74|6%<br>5%|34.98 <br>16.79|100%<br> %66|
3: |<br>|||.|..|Olorunsogo NIPP|<br>690|94.77|14%|73.82|100%|
4: |4,274.80MWh/h▼3.94%<br>Average Hourly Generation|||k|35080kV▲124%|Omotosho NIPP|500|42.26|8%|26.57|85%|
5: |<br>|Grid V|upper<br>oltage|46.50|..|Sapele GT NIPP|452|58.90|13%|46.34|<sup>100%</sup>|
6: |3,180.45GWh▼3.94%<br>Total Energy Generated||<br>lower|313.50kV|299.93kV▼4.33%|Ihovbor NIPP|450|11.02|2%|7.10|%L8|


In [34]:
grid_line_1_data = grid_lines[0].strip().split("|")
for line in grid_line_1_data:
  print(f"'{line}'")

''
'Grid-Connected Plants<br>26'
''
''
'Target'
'Actual'
'Plant'
'Capacity<br>Installed<br>(MW)'
'Available<br>Capacity<br>Average<br>(MW)<br>A<br>F'
'vailabilit<br>actor (%<br>Plant'
'y<br>)<br>Generation<br>(GWh)<br>Total<br>'
'Factor<br>Load<br>(%)'
''


In [35]:
# Isolate the grid only columns from the isolated grid rows
grid_data = []
for line in grid_lines:
  grid_line_data = line.split("|")[1:6]
  grid_data.append("|".join(grid_line_data))

for data in grid_data:
  print(data)

Grid-Connected Plants<br>26|||Target|Actual
4,459.41MW▼5.66%<br>Total Available Capacity|Grid Fr|upper <br>equency<br>lower|50.25Hz <br>4975Hz|50.62Hz▲0.70%<br>4903Hz▼145%
<br>|||.|..
4,274.80MWh/h▼3.94%<br>Average Hourly Generation|||k|35080kV▲124%
<br>|Grid V|upper<br>oltage|46.50|..
3,180.45GWh▼3.94%<br>Total Energy Generated||<br>lower|313.50kV|299.93kV▼4.33%


In [36]:
# Extract the important grid data from the isolated grid cells

grid_values = []
for line in grid_data:
  cells = line.split("|")
  grid_values.append(cells[-1].strip())

for value in grid_values:
  print(value)

Actual
50.62Hz▲0.70%<br>4903Hz▼145%
..
35080kV▲124%
..
299.93kV▼4.33%


In [37]:
grid_idx = len(df_grid)

df_grid.loc[grid_idx, ['source_id', 'reporting_month']] = [
    'SRC_0001', '2024-01']

In [38]:
df_grid.head()

,source_id,reporting_month,lower_voltage,upper_voltage,lower_frequency,upper_frequency
0,SRC_0021,2025-08,300.65kV,350.77kV,49.37Hz,50.78Hz
1,SRC_0001,2024-01,NaN,NaN,NaN,NaN


In [39]:
freq_pattern = r"(\d+\.*\d+Hz)\S+\>(\d+\.*\d+Hz)"
freq_data = re.search(freq_pattern, grid_values[1])
if freq_data:
  lower_freq = freq_data.group(2)
  upper_freq = freq_data.group(1)
  df_grid.loc[grid_idx, ['lower_frequency', 'upper_frequency']] = [lower_freq, upper_freq]

In [40]:
df_grid.head()

,source_id,reporting_month,lower_voltage,upper_voltage,lower_frequency,upper_frequency
0,SRC_0021,2025-08,300.65kV,350.77kV,49.37Hz,50.78Hz
1,SRC_0001,2024-01,NaN,NaN,4903Hz,50.62Hz


In [41]:
voltage_pattern = r"(\d+\.*\d+kV)\S+"

upper_voltage_data = re.search(voltage_pattern, grid_values[3])
lower_voltage_data = re.search(voltage_pattern, grid_values[5])
df_grid.loc[grid_idx, ['lower_voltage', 'upper_voltage']] = [
    lower_voltage_data.group(1), upper_voltage_data.group(1)]

In [42]:
df_grid.head()

,source_id,reporting_month,lower_voltage,upper_voltage,lower_frequency,upper_frequency
0,SRC_0021,2025-08,300.65kV,350.77kV,49.37Hz,50.78Hz
1,SRC_0001,2024-01,299.93kV,35080kV,4903Hz,50.62Hz


## Initial extraction for the plant data for old document formats

In [43]:
for line_number, line in enumerate(md_old_text.splitlines()):
  print(f"{line_number}: {line}")

0: |Nigerian Electricity<br>Regulatory Commission<br>NERC||Operati<br>of|onal Pe<br>Power|rformance<br>Plants||||F|ACTSH<br>January, 2|EET<br>024|
1: |---|---|---|---|---|---|---|---|---|---|---|
2: |Energy Generation||Grid P|erforman|ce|O|ther Grid|Connec<br>|ted Pla|nts||
3: |Grid-Connected Plants<br>26|||Target|Actual|Plant|Capacity<br>Installed<br>(MW)|Available<br>Capacity<br>Average<br>(MW)<br>A<br>F|vailabilit<br>actor (%<br>Plant|y<br>)<br>Generation<br>(GWh)<br>Total<br>|Factor<br>Load<br>(%)|
4: |4,459.41MW▼5.66%<br>Total Available Capacity|Grid Fr|upper <br>equency<br>lower|50.25Hz <br>4975Hz|50.62Hz▲0.70%<br>4903Hz▼145%|Afam IV - V<br>Sapele ST|726<br>720|43.97<br>32.74|6%<br>5%|34.98 <br>16.79|100%<br> %66|
5: |<br>|||.|..|Olorunsogo NIPP|<br>690|94.77|14%|73.82|100%|
6: |4,274.80MWh/h▼3.94%<br>Average Hourly Generation|||k|35080kV▲124%|Omotosho NIPP|500|42.26|8%|26.57|85%|
7: |<br>|Grid V|upper<br>oltage|46.50|..|Sapele GT NIPP|452|58.90|13%|46.34|<sup>100%</sup>|
8: |3,1

In [60]:
data_lines = md_old_text.splitlines()[4:20]
data_line = data_lines[1].split("|")
for line in data_line: print(f"'{line}'")

# plant_data = []
# for line in data_lines:
#   cells = line.split("|")

''
'<br>'
''
''
'.'
'..'
'Olorunsogo NIPP'
'<br>690'
'94.77'
'14%'
'73.82'
'100%'
''


In [61]:
# Isolate the plant data for the other grid connected plants

other_plant_data = []
for line_number, line in enumerate(data_lines):
  cells = line.split("|")[6:12]
  other_plant_data.append("|".join(cells))

for data in other_plant_data:
  print(data)

Afam IV - V<br>Sapele ST|726<br>720|43.97<br>32.74|6%<br>5%|34.98 <br>16.79|100%<br> %66
Olorunsogo NIPP|<br>690|94.77|14%|73.82|100%
Omotosho NIPP|500|42.26|8%|26.57|85%
Sapele GT NIPP|452|58.90|13%|46.34|<sup>100%</sup>
Ihovbor NIPP|450|11.02|2%|7.10|%L8
Geregu NIPP|435|119.25|27%|88.91|100%
Omotosho|304|91.51|%0%|69.12|100%
Olorunsogo|304|83.54|%%|63.37|100%
Ibom Power|191|145.00|76%|61.78|57%
River IPP|180|113.65|63%|85.94|100%
Omoku|150|52.20|35%|49.82|100%
Trans Amadi|100|35.81|36%|32.85|100%
Paras|96|93.50|%L6|54.24|85%
Taoex Ener|60|873|15%|838|100%
p gy<br>||.||.|
Dadin Kowa|40|34.00|85%|25.48|100%


In [95]:
# Clean plant data for other grid connected plants

other_plants_pattern = (
  r"(?P<plant_name>[^|]*)\|"
  r"(?P<installed_capacity>[^|]*)\|"
  r"(?P<available_capacity>[^|]*)\|"
  r"(?P<availability_factor>[^|]*)\|"
  r"(?P<total_generation>[^|]*)\|"
  r"(?P<load_factor>[^|]*)"
)

rows = []

for line_number, line in enumerate(other_plant_data, start=4):
  line_match = re.search(other_plants_pattern, line)
  split_rows = []
  if line_match:
    for key, val in line_match.groupdict().items():
      val_split = list(filter(lambda x: bool(x.strip()), val.split("<br>")))
      for idx, value in enumerate(val_split):
        clean_value = re.sub(r"<[^>]*>", "", value.strip())
        if (idx >= len(split_rows)):
          split_rows.append({
            "source_id": "SRC_0001",
            "line_number": line_number,
            "reporting_month": "2024-01",
            key: clean_value
          })
        else:
          split_rows[idx][key] = clean_value
    rows.extend(split_rows)

df_plant = pd.concat([df_plant, pd.DataFrame(rows)], ignore_index=True)

In [96]:
df_plant

,source_id,line_number,reporting_month,plant_name,installed_capacity,available_capacity,availability_factor,average_generation,total_generation,load_factor
0,SRC_0021,14,2025-08,Egbin_1,1320MW,690,52%,640MWh/h,NaN,93%
1,SRC_0021,14,2025-08,Detl,900MW,507,56%,429MWh/h,NaN,85%
2,SRC_0021,14,2025-08,Kainji_1,760MW,397,52%,374MWh/h,NaN,94%
3,SRC_0021,14,2025-08,Zungeru_1,700MW,681,97%,292MWh/h,NaN,43%
4,SRC_0021,14,2025-08,Afam_2,650MW,203,31%,196MWh/h,NaN,97%
5,SRC_0021,14,2025-08,Shiroro_1,600MW,435,73%,259MWh/h,NaN,60%
6,SRC_0021,14,2025-08,Jebba_1,578MW,472,82%,344MWh/h,NaN,73%
7,SRC_0021,14,2025-08,Okpai_1,480MW,319,66%,249MWh/h,NaN,78%
8,SRC_0021,14,2025-08,Ihovbor_2,461MW,332,72%,302MWh/h,NaN,91%
9,SRC_0021,14,2025-08,Geregu_1,435MW,235,54%,167MWh/h,NaN,71%


In [97]:
# Isolate the top 10 largest energy producers data

watermark_pattern = r"100%<br>80<br>60<br>40<br>20<br><br>|40<br>60<br>80<br>100%<br>|0<br>0<br>|<br>"

top_10_lines = md_old_text.splitlines()[11:21]
top_10_plant_data = []

for line in top_10_lines:
  cells = line.split("|")[1:6]
  relevant_cells = [cells[0], cells[1], f"{cells[3].strip()}{cells[4].lstrip()}"]
  cleaned_cells = [re.sub(watermark_pattern, "", cell).strip() for cell in relevant_cells]
  for i in range(len(cleaned_cells) - 1):
    if cleaned_cells[i].endswith("MW") and cleaned_cells[i + 1].startswith(")"):
      cleaned_cells[i] += ")"
      cleaned_cells[i + 1] = cleaned_cells[i + 1].lstrip()[1:]
  top_10_plant_data.append("|".join(cleaned_cells))

for data in top_10_plant_data:
  print(data)

37% (488/1320MW)|Ebi ST|99% (360GWh)
39% (353/900MW)|gnDelta GS|98% (256GWh)
58% (438/760MW)|Kainji|98% (318GWh)
54% (349/650MW)|Afam VI|100% (262GWh)
34% (210/625MW)|Odudknn|98% (153GWh)
52% (315/600MW)|Shiroro|98% (229GWh)
57% (327/570MW)|Jebba|90% (220GWh)
46% (223/480MW)|Okpai|90% (150GWh)
98% (451/461MW)|Azura IPP|90% (301GWh)
56% (244/435MW)|Geregu|100% (181GWh)


In [98]:
# Clean the dirty data for top 10 largest energy producers
top_10_pattern = (
  r"(?P<availability_factor>\d+%)\s*\("
  r"(?P<available_capacity>\d+)/"
  r"(?P<installed_capacity>\d+MW)\)\s*\|"
  r"(?P<plant_name>[^|]*)\|"
  r"(?P<load_factor>\d+%)\s*\("
  r"(?P<total_generation>\d+GWh)\)"
)

rows = []

for line_number, line in enumerate(top_10_plant_data, start=11):
  line_match = re.search(top_10_pattern, line)
  if line_match:
    rows.append({
      'source_id': 'SRC_0001',
      'line_number': line_number,
      "reporting_month": '2024-01',
      **line_match.groupdict()
    })

df_plant = pd.concat([df_plant, pd.DataFrame(rows)], ignore_index=True)

In [99]:
df_plant

,source_id,line_number,reporting_month,plant_name,installed_capacity,available_capacity,availability_factor,average_generation,total_generation,load_factor
0,SRC_0021,14,2025-08,Egbin_1,1320MW,690,52%,640MWh/h,NaN,93%
1,SRC_0021,14,2025-08,Detl,900MW,507,56%,429MWh/h,NaN,85%
2,SRC_0021,14,2025-08,Kainji_1,760MW,397,52%,374MWh/h,NaN,94%
3,SRC_0021,14,2025-08,Zungeru_1,700MW,681,97%,292MWh/h,NaN,43%
4,SRC_0021,14,2025-08,Afam_2,650MW,203,31%,196MWh/h,NaN,97%
5,SRC_0021,14,2025-08,Shiroro_1,600MW,435,73%,259MWh/h,NaN,60%
6,SRC_0021,14,2025-08,Jebba_1,578MW,472,82%,344MWh/h,NaN,73%
7,SRC_0021,14,2025-08,Okpai_1,480MW,319,66%,249MWh/h,NaN,78%
8,SRC_0021,14,2025-08,Ihovbor_2,461MW,332,72%,302MWh/h,NaN,91%
9,SRC_0021,14,2025-08,Geregu_1,435MW,235,54%,167MWh/h,NaN,71%


In [1]:
from generation_analysis import OldFormatExtractor
from pathlib import Path

md_old_path = Path(
    "../data/interim/markdown/operational-performance-jan-2024_converted.md")
md_old_text = md_old_path.read_text(encoding="utf-8")

In [2]:
extractor = OldFormatExtractor(md_old_text, 'SRC_0001', '2024-01')
result = extractor.extract()

In [3]:
result[0]

,source_id,reporting_month,upper_frequency,lower_frequency,upper_voltage,lower_voltage
0,SRC_0001,2024-01,50.62Hz,4903Hz,35080kV,299.93kV


In [4]:
result[1]

,source_id,line_number,reporting_month,availability_factor,available_capacity,installed_capacity,plant_name,load_factor,total_generation
0,SRC_0001,11,2024-01,37%,488,1320MW,Ebi ST,99%,360GWh
1,SRC_0001,12,2024-01,39%,353,900MW,gnDelta GS,98%,256GWh
2,SRC_0001,13,2024-01,58%,438,760MW,Kainji,98%,318GWh
3,SRC_0001,14,2024-01,54%,349,650MW,Afam VI,100%,262GWh
4,SRC_0001,15,2024-01,34%,210,625MW,Odudknn,98%,153GWh
5,SRC_0001,16,2024-01,52%,315,600MW,Shiroro,98%,229GWh
6,SRC_0001,17,2024-01,57%,327,570MW,Jebba,90%,220GWh
7,SRC_0001,18,2024-01,46%,223,480MW,Okpai,90%,150GWh
8,SRC_0001,19,2024-01,98%,451,461MW,Azura IPP,90%,301GWh
9,SRC_0001,20,2024-01,56%,244,435MW,Geregu,100%,181GWh
